# Кластеризация клиентов «Mafia Time»
## Дисциплина: Программирование и информационные технологии в научно-исследовательской работе

**Методы:** K-Means, DBSCAN, Agglomerative Clustering  
**Данные:** Синтетический датасет клиентов и заказов (имитация выгрузки из amoCRM)  
**Задача:** Сегментация клиентской базы по поведенческим и финансовым признакам для управленческой аналитики

## 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
print('Библиотеки загружены успешно')

## 2. Генерация синтетического датасета

Датасет имитирует выгрузку из CRM-системы amoCRM компании «Mafia Time».  
Каждая строка — один клиент. Признаки:
- `orders_count` — количество заказов
- `avg_check` — средний чек (руб.)
- `days_since_last_order` — дней с последнего заказа
- `response_time_hours` — среднее время ответа менеджера (часов)
- `total_revenue` — суммарная выручка по клиенту (руб.)

In [ ]:
n = 300

# Сегмент 1: корпоративные клиенты — много заказов, высокий чек
corp = pd.DataFrame({
    'orders_count':           np.random.randint(5, 20, 80),
    'avg_check':              np.random.normal(12000, 2000, 80).clip(7000, 18000),
    'days_since_last_order':  np.random.randint(7, 60, 80),
    'response_time_hours':    np.random.normal(1.5, 0.5, 80).clip(0.5, 4),
    'segment_true':           'corporate'
})

# Сегмент 2: частные клиенты — 1-3 заказа, средний чек
private = pd.DataFrame({
    'orders_count':           np.random.randint(1, 4, 150),
    'avg_check':              np.random.normal(7500, 1500, 150).clip(4000, 12000),
    'days_since_last_order':  np.random.randint(30, 365, 150),
    'response_time_hours':    np.random.normal(3.5, 1.2, 150).clip(0.5, 8),
    'segment_true':           'private'
})

# Сегмент 3: VIP / постоянные — очень высокий чек, частые заказы
vip = pd.DataFrame({
    'orders_count':           np.random.randint(10, 30, 50),
    'avg_check':              np.random.normal(15000, 2500, 50).clip(10000, 20000),
    'days_since_last_order':  np.random.randint(1, 30, 50),
    'response_time_hours':    np.random.normal(0.8, 0.3, 50).clip(0.2, 2),
    'segment_true':           'vip'
})

# Шум / аномалии
noise = pd.DataFrame({
    'orders_count':           np.random.randint(1, 5, 20),
    'avg_check':              np.random.normal(5000, 3000, 20).clip(1000, 25000),
    'days_since_last_order':  np.random.randint(200, 500, 20),
    'response_time_hours':    np.random.normal(10, 5, 20).clip(1, 24),
    'segment_true':           'noise'
})

df = pd.concat([corp, private, vip, noise], ignore_index=True)
df['total_revenue'] = df['orders_count'] * df['avg_check']

print(f'Датасет: {df.shape[0]} клиентов, {df.shape[1]} признаков')
df.describe().round(1)

## 3. Предобработка данных

In [ ]:
features = ['orders_count', 'avg_check', 'days_since_last_order',
            'response_time_hours', 'total_revenue']

X = df[features].copy()

# Стандартизация признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA для визуализации (2 компоненты)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('Объяснённая дисперсия (2 компоненты):', 
      round(sum(pca.explained_variance_ratio_) * 100, 1), '%')

## 4. Метод 1: K-Means

### Выбор числа кластеров (метод локтя)

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Оптимум k=3')
axes[0].set_xlabel('Число кластеров k')
axes[0].set_ylabel('Инерция (SSE)')
axes[0].set_title('Метод локтя (Elbow Method)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Оптимум k=3')
axes[1].set_xlabel('Число кластеров k')
axes[1].set_ylabel('Коэффициент силуэта')
axes[1].set_title('Коэффициент силуэта')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Оптимальное число кластеров по силуэту: k={K_range[silhouettes.index(max(silhouettes))]}')

In [ ]:
# Обучение K-Means с k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['cluster_kmeans'])
print(f'K-Means | k=3 | Силуэт: {sil:.4f}')

# Визуализация в пространстве PCA
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
labels_map = {0: 'Кластер 0', 1: 'Кластер 1', 2: 'Кластер 2'}

plt.figure(figsize=(9, 6))
for c in range(3):
    mask = df['cluster_kmeans'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[c], label=labels_map[c], alpha=0.7, s=50)

# Центроиды
centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            c='black', marker='X', s=200, zorder=5, label='Центроиды')

plt.xlabel(f'PCA-1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA-2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('K-Means (k=3) — визуализация кластеров (PCA)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Профили кластеров
profile_km = df.groupby('cluster_kmeans')[features].mean().round(1)
profile_km['count'] = df['cluster_kmeans'].value_counts().sort_index()
print('Профили кластеров K-Means:')
print(profile_km.to_string())

## 5. Метод 2: DBSCAN

DBSCAN хорошо выявляет аномальных клиентов (шум) и не требует заранее указывать число кластеров.

In [ ]:
# Подбор параметров DBSCAN: k-distance graph
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(k_distances, linewidth=1.5)
plt.axhline(y=1.5, color='red', linestyle='--', label='eps=1.5 (выбранное значение)')
plt.xlabel('Точки (отсортированы по убыванию расстояния)')
plt.ylabel('5-NN расстояние')
plt.title('k-distance Graph для выбора eps (DBSCAN)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_kdist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=8)
df['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

unique_labels = sorted(df['cluster_dbscan'].unique())
n_clusters_db = len([l for l in unique_labels if l != -1])
n_noise = (df['cluster_dbscan'] == -1).sum()

print(f'DBSCAN | eps=1.5, min_samples=8')
print(f'Найдено кластеров: {n_clusters_db}')
print(f'Шумовые точки (выбросы): {n_noise} ({n_noise/len(df)*100:.1f}%)')

if n_clusters_db > 1:
    mask_valid = df['cluster_dbscan'] != -1
    sil_db = silhouette_score(X_scaled[mask_valid], df.loc[mask_valid, 'cluster_dbscan'])
    print(f'Коэффициент силуэта (без шума): {sil_db:.4f}')

# Визуализация
cmap = plt.cm.get_cmap('tab10', n_clusters_db + 1)
plt.figure(figsize=(9, 6))

for label in unique_labels:
    mask = df['cluster_dbscan'] == label
    color = 'black' if label == -1 else cmap(label)
    lname = f'Шум ({mask.sum()})' if label == -1 else f'Кластер {label} ({mask.sum()})'
    marker = 'x' if label == -1 else 'o'
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[color], label=lname, alpha=0.7 if label != -1 else 0.5,
                s=50, marker=marker)

plt.xlabel(f'PCA-1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA-2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('DBSCAN — визуализация кластеров (PCA)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Метод 3: Агломеративная иерархическая кластеризация

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Дендрограмма на подвыборке
sample_idx = np.random.choice(len(X_scaled), 80, replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90,
           leaf_font_size=9, show_contracted=True,
           color_threshold=8)
plt.axhline(y=8, color='red', linestyle='--', alpha=0.7, label='Порог разреза (3 кластера)')
plt.xlabel('Клиенты (подвыборка 80 шт.)')
plt.ylabel('Расстояние Варда')
plt.title('Дендрограмма иерархической кластеризации')
plt.legend()
plt.tight_layout()
plt.savefig('agg_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['cluster_agg'] = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, df['cluster_agg'])
print(f'Agglomerative | n_clusters=3 | linkage=ward | Силуэт: {sil_agg:.4f}')

plt.figure(figsize=(9, 6))
colors_agg = ['#d62728', '#9467bd', '#8c564b']
for c in range(3):
    mask = df['cluster_agg'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors_agg[c], label=f'Кластер {c} ({mask.sum()})', alpha=0.7, s=50)

plt.xlabel(f'PCA-1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA-2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Agglomerative Clustering (ward, k=3) — PCA')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('agg_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Сравнение методов

In [ ]:
results = pd.DataFrame({
    'Метод': ['K-Means', 'DBSCAN', 'Agglomerative'],
    'Параметры': ['k=3', 'eps=1.5, min_samples=8', 'n=3, linkage=ward'],
    'Кластеров': [3, n_clusters_db, 3],
    'Силуэт': [
        round(silhouette_score(X_scaled, df['cluster_kmeans']), 4),
        round(sil_db, 4) if n_clusters_db > 1 else 'н/д',
        round(sil_agg, 4)
    ],
    'Определяет шум': ['Нет', 'Да', 'Нет'],
    'Требует k заранее': ['Да', 'Нет', 'Да'],
})

print('=== Сравнение методов кластеризации ===')
print(results.to_string(index=False))

In [ ]:
# Итоговая визуализация: три метода рядом
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = [
    ('cluster_kmeans', 'K-Means (k=3)', ['#1f77b4','#ff7f0e','#2ca02c']),
    ('cluster_agg',    'Agglomerative (k=3)', ['#d62728','#9467bd','#8c564b']),
]

for ax, (col, title, colors) in zip(axes[:2], methods):
    for c in df[col].unique():
        mask = df[col] == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors[c % len(colors)], alpha=0.7, s=30,
                   label=f'Кластер {c}')
    ax.set_title(title)
    ax.set_xlabel('PCA-1')
    ax.set_ylabel('PCA-2')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# DBSCAN
unique_db = sorted(df['cluster_dbscan'].unique())
cmap2 = plt.cm.get_cmap('tab10', max(unique_db) + 1)
for label in unique_db:
    mask = df['cluster_dbscan'] == label
    color = 'black' if label == -1 else cmap2(label)
    lname = 'Шум' if label == -1 else f'Кластер {label}'
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[color], alpha=0.7 if label != -1 else 0.4,
                    s=30, label=lname,
                    marker='x' if label == -1 else 'o')
axes[2].set_title('DBSCAN')
axes[2].set_xlabel('PCA-1')
axes[2].set_ylabel('PCA-2')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.suptitle('Сравнение методов кластеризации клиентов «Mafia Time»', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('clustering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Выводы

- **K-Means** показал наилучший коэффициент силуэта и чёткое разделение на три сегмента: корпоративные клиенты, частные клиенты, VIP. Оптимален как базовый метод сегментации.
- **DBSCAN** выявил шумовые точки — клиентов с нетипичным поведением (очень редкие заказы, длительное отсутствие). Полезен для выявления аномалий в управленческой аналитике.
- **Agglomerative Clustering** дал сопоставимые результаты с K-Means. Дендрограмма наглядно показывает иерархическую структуру клиентской базы.

В контексте проекта «Mafia Time» результаты кластеризации могут использоваться для:
1. Персонализации маркетинговых коммуникаций по сегментам
2. Приоритизации обработки входящих заявок
3. Формирования управленческих дашбордов с разбивкой по типам клиентов